# EMMM: generating full-body human motion from EEG

In [ ]:
import torch
import clip
import models.vqvae as vqvae
from models.vqvae_sep import VQVAE_SEP
import models.t2m_trans as trans
import models.t2m_trans_uplow as trans_uplow
from tqdm import tqdm
import numpy as np
from exit.utils import visualize_2motions
import options.option_eeg2motion as option_trans
import sys 
import torch.nn.functional as F
from torch.utils import data
import torch.nn as nn
import scipy
import utils.utils_model as utils_model
import os
from einops import rearrange, repeat
from dataset import dataset_EM_train,dataset_EM_eval
from utils.eval_trans import calculate_R_precision,euclidean_distance_matrix,calculate_activation_statistics,calculate_diversity,calculate_frechet_distance
from utils.log_utils import setup_logger
from options.get_eval_option import get_opt
from models.evaluator_wrapper import EvaluatorModelWrapper
import random

In [ ]:
sys.argv = ['','--resume-pth','output/vq/2024-06-03-20-22-07_retrain/net_last.pth','--resume-trans','output/t2m/2024-06-04-09-29-20_trans_name_b128/net_last.pth']
args = option_trans.get_args_parser()
args.vq_dir = f'./output/vq/{args.vq_name}' #os.path.join("./dataset/KIT-ML" if args.dataname == 'kit' else "./dataset/HumanML3D", f'{args.vq_name}')
codebook_dir = f'{args.vq_dir}/codebook/'

dataset_opt_path = 'checkpoints/kit/Comp_v6_KLD005/opt.txt' if args.dataname == 'kit' else 'checkpoints/t2m/Comp_v6_KLD005/opt.txt'

wrapper_opt = get_opt(dataset_opt_path, torch.device('cuda'))
eval_wrapper = EvaluatorModelWrapper(wrapper_opt)

In [ ]:
def set_seed(seed):
    random.seed(seed)                     
    np.random.seed(seed)                   
    torch.manual_seed(seed)               
    torch.cuda.manual_seed(seed)           
    torch.cuda.manual_seed_all(seed)       
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False 

set_seed(args.seed)

# Loda Text Motion models

In [ ]:
##### ---- CLIP ---- #####
clip_model, clip_preprocess = clip.load("ViT-B/32", device=torch.device('cuda'), jit=False)  # Must set jit=False for training
clip.model.convert_weights(clip_model)  # Actually this line is unnecessary since clip by default already on float16
clip_model.eval()
for p in clip_model.parameters():
    p.requires_grad = False

# https://github.com/openai/CLIP/issues/111
class TextCLIP(torch.nn.Module):
    def __init__(self, model) :
        super(TextCLIP, self).__init__()
        self.model = model
        
    def forward(self,text):
        with torch.no_grad():
            word_emb = self.model.token_embedding(text).type(self.model.dtype)
            word_emb = word_emb + self.model.positional_embedding.type(self.model.dtype)
            word_emb = word_emb.permute(1, 0, 2)  # NLD -> LND
            word_emb = self.model.transformer(word_emb)
            word_emb = self.model.ln_final(word_emb).permute(1, 0, 2).float()
            enctxt = self.model.encode_text(text).float()
        return enctxt, word_emb
clip_model = TextCLIP(clip_model)

def get_vqvae(args, is_upper_edit):
    if not is_upper_edit:
        return vqvae.HumanVQVAE(args, ## use args to define different parameters in different quantizers
                            args.nb_code,
                            args.code_dim,
                            args.output_emb_width,
                            args.down_t,
                            args.stride_t,
                            args.width,
                            args.depth,
                            args.dilation_growth_rate)
    else:
        return VQVAE_SEP(args, ## use args to define different parameters in different quantizers
                        args.nb_code,
                        args.code_dim,
                        args.output_emb_width,
                        args.down_t,
                        args.stride_t,
                        args.width,
                        args.depth,
                        args.dilation_growth_rate,
                        moment={'mean': torch.from_numpy(args.mean).cuda().float(), 
                            'std': torch.from_numpy(args.std).cuda().float()},
                        sep_decoder=True)

def get_maskdecoder(args, vqvae, is_upper_edit):
    tranformer = trans if not is_upper_edit else trans_uplow
    return tranformer.Text2Motion_Transformer(vqvae,
                                num_vq=args.nb_code, 
                                embed_dim=args.embed_dim_gpt, 
                                clip_dim=args.clip_dim, 
                                block_size=args.block_size, 
                                num_layers=args.num_layers, 
                                num_local_layer=args.num_local_layer, 
                                n_head=args.n_head_gpt, 
                                drop_out_rate=args.drop_out_rate, 
                                fc_rate=args.ff_rate)

class MMM(torch.nn.Module):
    def __init__(self, args=None, is_upper_edit=False):
        super().__init__()
        self.is_upper_edit = is_upper_edit


        args.dataname = args.dataset_name = 't2m'

        self.vqvae = get_vqvae(args, is_upper_edit)
        ckpt = torch.load(args.resume_pth, map_location='cpu')
        self.vqvae.load_state_dict(ckpt['net'], strict=True)
        if is_upper_edit:
            class VQVAE_WRAPPER(torch.nn.Module):
                def __init__(self, vqvae) :
                    super().__init__()
                    self.vqvae = vqvae
                    
                def forward(self, *args, **kwargs):
                    return self.vqvae(*args, **kwargs)
            self.vqvae = VQVAE_WRAPPER(self.vqvae)
        self.vqvae.eval()
        self.vqvae.cuda()

        self.maskdecoder = get_maskdecoder(args, self.vqvae, is_upper_edit)
        ckpt = torch.load(args.resume_trans, map_location='cpu')
        self.maskdecoder.load_state_dict(ckpt['trans'], strict=True)
        self.maskdecoder.eval()
        self.maskdecoder.cuda()

    def forward(self, text, lengths=-1, rand_pos=True):
        b = len(text)
        feat_clip_text = clip.tokenize(text, truncate=True).cuda()
        feat_clip_text, word_emb = clip_model(feat_clip_text)
        # feat_clip_text_null = clip.tokenize([''], truncate=True).cuda()
        # feat_clip_text_null, word_emb_null = clip_model(feat_clip_text_null)
        # index_motion = self.maskdecoder(feat_clip_text, word_emb_null, type="sample", m_length=lengths, rand_pos=rand_pos, if_test=False)
        index_motion,logits = self.maskdecoder(feat_clip_text, word_emb, type="sample", m_length=lengths, rand_pos=rand_pos, if_test=False)

        m_token_length = torch.ceil((lengths)/4).int()
        pred_pose_all = torch.zeros((b, 196, 263)).cuda()
        for k in range(b):
            pred_pose = self.vqvae(index_motion[k:k+1, :m_token_length[k]], type='decode')
            pred_pose_all[k:k+1, :int(lengths[k].item())] = pred_pose
        return pred_pose_all,index_motion,logits

    def inbetween_eval(self, base_pose, m_length, start_f, end_f, inbetween_text):
        bs, seq = base_pose.shape[:2]
        tokens = -1*torch.ones((bs, 50), dtype=torch.long).cuda()
        m_token_length = torch.ceil((m_length)/4).int()
        start_t = torch.round((start_f)/4).int()
        end_t = torch.round((end_f)/4).int()

        for k in range(bs):
            index_motion = self.vqvae(base_pose[k:k+1, :m_length[k]].cuda(), type='encode')
            tokens[k, :start_t[k]] = index_motion[0][:start_t[k]]
            tokens[k, end_t[k]:m_token_length[k]] = index_motion[0][end_t[k]:m_token_length[k]]

        text = clip.tokenize(inbetween_text, truncate=True).cuda()
        feat_clip_text, word_emb_clip = clip_model(text)

        mask_id = self.maskdecoder.num_vq + 2
        tokens[tokens==-1] = mask_id
        inpaint_index = self.maskdecoder(feat_clip_text, word_emb_clip, type="sample", m_length=m_length.cuda(), token_cond=tokens)

        pred_pose_eval = torch.zeros((bs, seq, base_pose.shape[-1])).cuda()
        for k in range(bs):
            pred_pose = self.vqvae(inpaint_index[k:k+1, :m_token_length[k]], type='decode')
            pred_pose_eval[k:k+1, :int(m_length[k].item())] = pred_pose
        return pred_pose_eval

    def long_range(self, text, lengths, num_transition_token=2, output='concat', index_motion=None):
        b = len(text)
        feat_clip_text = clip.tokenize(text, truncate=True).cuda()
        feat_clip_text, word_emb = clip_model(feat_clip_text)
        if index_motion is None:
            index_motion = self.maskdecoder(feat_clip_text, word_emb, type="sample", m_length=lengths, rand_pos=False)

        m_token_length = torch.ceil((lengths)/4).int()
        if output == 'eval':
            frame_length = m_token_length * 4
            m_token_length = m_token_length.clone()
            m_token_length = m_token_length - 2*num_transition_token
            m_token_length[[0,-1]] += num_transition_token # first and last have transition only half
        
        half_token_length = (m_token_length/2).int()
        idx_full_len = half_token_length >= 24
        half_token_length[idx_full_len] = half_token_length[idx_full_len] - 1

        mask_id = self.maskdecoder.num_vq + 2
        tokens = -1*torch.ones((b-1, 50), dtype=torch.long).cuda()
        transition_train_length = []
        
        for i in range(b-1):
            if output == 'concat':
                i_index_motion = index_motion[i]
                i1_index_motion = index_motion[i+1]
            if output == 'eval':
                if i == 0:
                    i_index_motion = index_motion[i, :m_token_length[i]]
                else:
                    i_index_motion = index_motion[i, num_transition_token:m_token_length[i] + num_transition_token]
                if i == b-1:
                    i1_index_motion = index_motion[i+1, :m_token_length[i+1]]
                else:
                    i1_index_motion = index_motion[i+1, 
                                                num_transition_token:m_token_length[i+1] + num_transition_token]
            left_end = half_token_length[i]
            right_start = left_end + num_transition_token
            end = right_start + half_token_length[i+1]

            tokens[i, :left_end] = i_index_motion[m_token_length[i]-left_end: m_token_length[i]]
            tokens[i, left_end:right_start] = mask_id
            tokens[i, right_start:end] = i1_index_motion[:half_token_length[i+1]]
            transition_train_length.append(end)
        transition_train_length = torch.tensor(transition_train_length).to(index_motion.device)
        text = clip.tokenize(text[:-1], truncate=True).cuda()
        feat_clip_text, word_emb_clip = clip_model(text)
        inpaint_index = self.maskdecoder(feat_clip_text, word_emb_clip, type="sample", m_length=transition_train_length*4, token_cond=tokens, max_steps=1)
        
        if output == 'concat':
            all_tokens = []
            for i in range(b-1):
                all_tokens.append(index_motion[i, :m_token_length[i]])
                all_tokens.append(inpaint_index[i, tokens[i] == mask_id])
            all_tokens.append(index_motion[-1, :m_token_length[-1]])
            all_tokens = torch.cat(all_tokens).unsqueeze(0)
            pred_pose = self.vqvae(all_tokens, type='decode')
            return pred_pose
        elif output == 'eval':
            all_tokens = []
            for i in range(b):
                motion_token = index_motion[i, :m_token_length[i]]
                if i == 0:
                    first_current_trans_tok = inpaint_index[i, tokens[i] == mask_id]
                    all_tokens.append(motion_token)
                    all_tokens.append(first_current_trans_tok)
                else:
                    if i < b-1:
                        first_current_trans_tok = inpaint_index[i, tokens[i] == mask_id]
                        all_tokens.append(motion_token)
                        all_tokens.append(first_current_trans_tok)
                    else:
                        all_tokens.append(motion_token)
            all_tokens = torch.cat(all_tokens)
            pred_pose_concat = self.vqvae(all_tokens.unsqueeze(0), type='decode')
            
            trans_frame = num_transition_token*4
            pred_pose = torch.zeros((b, 196, 263)).cuda()
            current_point = 0
            for i in range(b):
                if i == 0:
                    start_f = torch.tensor(0)
                    end_f = frame_length[i]
                else:
                    start_f = current_point - trans_frame
                    end_f = start_f + frame_length[i]
                current_point = end_f
                pred_pose[i, :frame_length[i]] = pred_pose_concat[0, start_f: end_f]
            return pred_pose

    def upper_edit(self, pose, m_length, upper_text, lower_mask=None):
        pose = pose.clone().cuda().float() # bs, nb_joints, joints_dim, seq_len
        m_tokens_len = torch.ceil((m_length)/4)
        bs, seq = pose.shape[:2]
        max_motion_length = int(seq/4) + 1
        mot_end_idx = self.vqvae.vqvae.num_code
        mot_pad_idx = self.vqvae.vqvae.num_code + 1
        mask_id = self.vqvae.vqvae.num_code + 2
        target_lower = []
        for k in range(bs):
            target = self.vqvae(pose[k:k+1, :m_length[k]], type='encode')
            if m_tokens_len[k]+1 < max_motion_length:
                target = torch.cat([target, 
                                    torch.ones((1, 1, 2), dtype=int, device=target.device) * mot_end_idx, 
                                    torch.ones((1, max_motion_length-1-m_tokens_len[k].int().item(), 2), dtype=int, device=target.device) * mot_pad_idx], axis=1)
            else:
                target = torch.cat([target, 
                                    torch.ones((1, 1, 2), dtype=int, device=target.device) * mot_end_idx], axis=1)
            target_lower.append(target[..., 1])
        target_lower = torch.cat(target_lower, axis=0)

        ### lower mask ###
        if lower_mask is not None:
            lower_mask = torch.cat([lower_mask, torch.zeros(bs, 1, dtype=int)], dim=1).bool()
            target_lower_masked = target_lower.clone()
            target_lower_masked[lower_mask] = mask_id
            select_end = target_lower == mot_end_idx
            target_lower_masked[select_end] = target_lower[select_end]
        else:
            target_lower_masked = target_lower
        ##################

        pred_len = m_length.cuda()
        pred_tok_len = m_tokens_len
        pred_pose_eval = torch.zeros((bs, seq, pose.shape[-1])).cuda()

        # __upper_text__ = ['A man punches with right hand.'] * 32
        text = clip.tokenize(upper_text, truncate=True).cuda()
        feat_clip_text, word_emb_clip = clip_model(text)
        # index_motion = trans_encoder(feat_clip_text, idx_lower=target_lower_masked, word_emb=word_emb_clip, type="sample", m_length=pred_len, rand_pos=True, CFG=-1)
        index_motion = self.maskdecoder(feat_clip_text, target_lower_masked, word_emb_clip, type="sample", m_length=pred_len, rand_pos=True)
        for i in range(bs):
            all_tokens = torch.cat([
                index_motion[i:i+1, :int(pred_tok_len[i].item()), None],
                target_lower[i:i+1, :int(pred_tok_len[i].item()), None]
            ], axis=-1)
            pred_pose = self.vqvae(all_tokens, type='decode')
            pred_pose_eval[i:i+1, :int(pred_len[i].item())] = pred_pose

        return pred_pose_eval

sys.argv = ['','--resume-pth','output/vq/2024-06-03-20-22-07_retrain/net_last.pth','--resume-trans','output/t2m/2024-06-04-09-29-20_trans_name_b128/net_last.pth']
args = option_trans.get_args_parser()
mmm = MMM(args).cuda()

In [ ]:
args.vq_dir = f'./output/vq/{args.vq_name}' #os.path.join("./dataset/KIT-ML" if args.dataname == 'kit' else "./dataset/HumanML3D", f'{args.vq_name}')
codebook_dir = f'{args.vq_dir}/codebook/'
from options.get_eval_option import get_opt
from models.evaluator_wrapper import EvaluatorModelWrapper

dataset_opt_path = 'checkpoints/kit/Comp_v6_KLD005/opt.txt' if args.dataname == 'kit' else 'checkpoints/t2m/Comp_v6_KLD005/opt.txt'

wrapper_opt = get_opt(dataset_opt_path, torch.device('cuda'))
eval_wrapper = EvaluatorModelWrapper(wrapper_opt)

# Load EEG-Text-Motion Dataset

In [ ]:
def get_motion_feat_t2m(dataset,mmm,eval_wrapper):
    all_motion_feat = []

    for i in range(len(dataset)):
        clip_text_train, train_motion, train_motion_len, eeg_train,subid,motion_key = dataset[i]

        train_motion = train_motion.long().cuda()

        true_pose = torch.zeros((1, 196, 263)).cuda()
        true_pose_ = mmm.vqvae(
            train_motion[:train_motion_len].unsqueeze(0),
            type='decode'
        )
        true_pose[:, :train_motion_len*4] = true_pose_

        movements = eval_wrapper.movement_encoder(true_pose[..., :-4]).detach()
        m_lens = torch.tensor([train_motion_len*4 // eval_wrapper.opt.unit_length]).cuda()

        motion_embedding = eval_wrapper.motion_encoder(movements, m_lens)

        motion_feat = motion_embedding  # [1, 512]

        all_motion_feat.append(motion_feat.cpu())

    all_motion_feat = torch.cat(all_motion_feat, dim=0)  # [N, 512]
    return all_motion_feat

def get_text_feat_clip(dataset,clip,clip_model):
    all_text_feat = []

    for i in range(len(dataset)):
        clip_text_train, train_motion, train_motion_len, eeg_train,subid,motion_key = dataset[i]

        feat_clip_text = clip.tokenize([clip_text_train], truncate=True).cuda()
        feat_clip_text, word_emb = clip_model(feat_clip_text)

        text_feat = feat_clip_text  # [1, 512]

        all_text_feat.append(text_feat.cpu())

    all_text_feat = torch.cat(all_text_feat, dim=0)  # [N, 512]
    return all_text_feat

def get_test_text_feat_clip(dataset,clip,clip_model):
    all_text_feat = []

    for i in range(len(dataset)):
        word_embeddings, pos_one_hots, clip_text, sent_len, pose, m_length, token, name, eeg,subid,motion_key = dataset[i]

        feat_clip_text = clip.tokenize([clip_text], truncate=True).cuda()
        feat_clip_text, word_emb = clip_model(feat_clip_text)

        text_feat = feat_clip_text  # [1, 512]

        all_text_feat.append(text_feat.cpu())

    all_text_feat = torch.cat(all_text_feat, dim=0)  # [N, 512]
    return all_text_feat

def get_test_motion_feat_t2m(dataset,eval_wrapper):
    all_motion_feat = []

    for i in range(len(dataset)):
        word_embeddings, pos_one_hots, clip_text, sent_len, pose, m_length, token, name, eeg,subid,motion_key = dataset[i]

        pose = torch.tensor(pose).unsqueeze(0).cuda()
        
        true_pose = torch.zeros((1, 196, 263)).cuda()
        true_pose[:, :pose.shape[1]] = pose
        
        movements = eval_wrapper.movement_encoder(true_pose[..., :-4]).detach()
        m_lens = torch.tensor([m_length // eval_wrapper.opt.unit_length]).cuda()
        motion_embedding = eval_wrapper.motion_encoder(movements, m_lens)

        motion_feat = motion_embedding  # [1, 512]

        all_motion_feat.append(motion_feat.cpu())

    all_motion_feat = torch.cat(all_motion_feat, dim=0)  # [N, 512]
    return all_motion_feat

@torch.no_grad()
def get_video_feat(dataset, video_feat_root):
    """
    dataset:
        (..., motion_key)

    video_feat_root:
        存放 .npy 的文件夹
    """

    all_video_feat = []

    for i in range(len(dataset)):

        clip_text_train, train_motion, train_motion_len, eeg_train, subid, motion_key = dataset[i]

        video_key = str(motion_key)
        video_path = os.path.join(video_feat_root, f"{video_key}.npy")

        if not os.path.exists(video_path):
            print(f"[Missing npy] {video_path}")
            continue

        # 1. load precomputed feature
        video_feat = np.load(video_path)  # (D,) or (T,D)

        video_feat = torch.tensor(video_feat).float().cuda()

        video_feat = video_feat.unsqueeze(0)  # [1, D]

        all_video_feat.append(video_feat.cpu())

    all_video_feat = torch.cat(all_video_feat, dim=0)

    return all_video_feat

@torch.no_grad()
def get_test_video_feat(dataset, video_feat_root):
    """
    dataset:
        (..., motion_key)

    video_feat_root:
        存放 .npy 的文件夹
    """

    all_video_feat = []

    for i in range(len(dataset)):

        word_embeddings, pos_one_hots, clip_text, sent_len, pose, m_length, token, name, eeg,subid,motion_key = dataset[i]

        video_key = str(motion_key)
        video_path = os.path.join(video_feat_root, f"{video_key}.npy")

        if not os.path.exists(video_path):
            print(f"[Missing npy] {video_path}")
            continue

        # 1. load precomputed feature
        video_feat = np.load(video_path)  # (D,) or (T,D)

        video_feat = torch.tensor(video_feat).float().cuda()

        video_feat = video_feat.unsqueeze(0)  # [1, D]

        all_video_feat.append(video_feat.cpu())

    all_video_feat = torch.cat(all_video_feat, dim=0)

    return all_video_feat

In [ ]:
args.eeg_data_root

In [ ]:
sub_n=len(args.eeg_data_root)
sub_n

In [ ]:
dataset = dataset_EM_train.EEG2MotionDataset(
        args.dataname,
        eeg_roots=args.eeg_data_root,
        eeg_name=args.eeg_data_name,
        tokenizer_name=codebook_dir,
        codebook_size=args.nb_code,
        unit_length=2**args.down_t,
        eeg_ch=args.eeg_ch
    )

In [ ]:
args.video_feat_root

In [ ]:
if  args.clip_target=='motion_t2m':
    target_feats = get_motion_feat_t2m(dataset,mmm,eval_wrapper).detach()
elif args.clip_target=='text_clip':
    target_feats = get_text_feat_clip(dataset,clip,clip_model).detach()
elif args.clip_target=='video_mae':
    target_feats = get_video_feat(dataset, video_feat_root=args.video_feat_root).detach()

if args.random_level:
    print("random shuffle")
    idx = torch.randperm(target_feats.size(0))
    target_feats = target_feats[idx]

train_with_feat_loader = dataset_EM_train.EEG2MotionLoaderWithFeat(args,base_dataset=dataset,target_feats=target_feats)
train_with_feat_loader_iter = dataset_EM_train.cycle(train_with_feat_loader)

In [ ]:
n_eeg_ch = dataset[0][3].shape[0]
n_eeg_ch

In [ ]:
val_dataset = dataset_EM_eval.EEG2MotionTestDataset(
    dataset_name=args.dataname,
    eeg_roots=args.eeg_data_root,
    eeg_name=args.eeg_data_name,
    test_avg=args.test_avg,
    dataset_type="val",
    eeg_ch=args.eeg_ch
)
if  args.clip_target=='motion_t2m':
    target_feats_val = get_test_motion_feat_t2m(val_dataset,eval_wrapper).detach()
elif args.clip_target=='text_clip':
    target_feats_val = get_test_text_feat_clip(val_dataset,clip,clip_model).detach()
elif args.clip_target=='video_mae':
    target_feats_val = get_test_video_feat(val_dataset, video_feat_root=args.video_feat_root).detach()
val_with_feat_loader = dataset_EM_eval.EEG2MotionLoaderWithFeat(base_dataset=val_dataset,target_feats=target_feats_val)
val_with_feat_loader_iter = dataset_EM_train.cycle(val_with_feat_loader)

In [ ]:
len(val_dataset)

In [ ]:
test_dataset = dataset_EM_eval.EEG2MotionTestDataset(
    dataset_name=args.dataname,
    eeg_roots=args.eeg_data_root,
    eeg_name=args.eeg_data_name,
    test_avg=args.test_avg,
    dataset_type="test",
    eeg_ch=args.eeg_ch
)
if  args.clip_target=='motion_t2m':
    target_feats_test = get_test_motion_feat_t2m(test_dataset,eval_wrapper).detach()
elif args.clip_target=='text_clip':
    target_feats_test = get_test_text_feat_clip(test_dataset,clip,clip_model).detach()
elif args.clip_target=='video_mae':
    target_feats_test = get_test_video_feat(test_dataset, video_feat_root=args.video_feat_root).detach()
test_with_feat_loader = dataset_EM_eval.EEG2MotionLoaderWithFeat(base_dataset=test_dataset,target_feats=target_feats_test)
test_with_feat_loader_iter = dataset_EM_train.cycle(test_with_feat_loader)

In [ ]:
len(test_dataset)

# EEG encoder

In [ ]:
if  args.clip_target=='motion_t2m':
    output_dim = 512 
elif args.clip_target=='text_clip':
    output_dim = 512 
elif args.clip_target=='video_mae':
    output_dim = 768

In [ ]:
from torch import Tensor
from einops.layers.torch import Rearrange, Reduce
import math
class AdaBN2d(nn.Module):
    """自适应批归一化：为每个被试维护独立的 BN 统计量与参数"""
    def __init__(self, num_features: int, n_subjects: int,
                 affine: bool = True, track_running_stats: bool = True):
        super().__init__()
        self.n_subjects = n_subjects
        self.bns = nn.ModuleList([
            nn.BatchNorm2d(num_features, affine=affine,
                           track_running_stats=track_running_stats)
            for _ in range(n_subjects)
        ])

    def forward(self, x: Tensor, subid: Tensor) -> Tensor:
        # 确保 subid 合法（0 ≤ subid < n_subjects）
        assert (subid >= 0).all() and (subid < self.n_subjects).all(), \
            f"subid must be in [0, {self.n_subjects-1}]"

        out = torch.empty_like(x)   
        for i in range(self.n_subjects):
            mask = (subid == i)
            if mask.any():
                out[mask] = self.bns[i](x[mask])
        return out


class PatchEmbedding(nn.Module):
    """PatchEmbedding with AdaBN for multiple subjects"""
    def __init__(self, emb_size: int = 40, n_subjects: int = 1,ch = 59):
        super().__init__()
        self.n_subjects = n_subjects

        self.conv1 = nn.Conv2d(1, emb_size, (1, 40), (1, 5))
        self.conv2 = nn.Conv2d(emb_size, emb_size, (ch, 1), (1, 1))
        self.ada_bn = AdaBN2d(emb_size, n_subjects)   # 替代原 BatchNorm2d
        self.elu = nn.ELU()
        self.pool = nn.AvgPool2d((1, 15), (1, 3))
        self.dropout = nn.Dropout(0.5)

        # 修正：使用 Rearrange 层
        self.projection = nn.Sequential(
            nn.Conv2d(emb_size, emb_size, (1, 1), stride=(1, 1)),
            Rearrange('b e h w -> b (h w) e')
        )

    def forward(self, x: Tensor, subid: Tensor) -> Tensor:
        # 输入 x 形状处理
        if x.dim() == 2:             # (B, T) -> (B, 1, 1, T)
            x = x.unsqueeze(1).unsqueeze(2)
        elif x.dim() == 3:           # (B, C, T) -> (B, 1, C, T)
            x = x.unsqueeze(1)

        x = self.conv1(x)
        x = self.conv2(x)
        x = self.ada_bn(x, subid)    # 按被试分别归一化
        x = self.elu(x)
        x = self.pool(x)
        x = self.dropout(x)
        x = self.projection(x)       # (B, patches, emb_size)
        return x


class MultiHeadAttention(nn.Module):
    def __init__(self, emb_size, num_heads, dropout):
        super().__init__()
        self.emb_size = emb_size
        self.num_heads = num_heads
        self.keys = nn.Linear(emb_size, emb_size)
        self.queries = nn.Linear(emb_size, emb_size)
        self.values = nn.Linear(emb_size, emb_size)
        self.att_drop = nn.Dropout(dropout)
        self.projection = nn.Linear(emb_size, emb_size)

    def forward(self, x: Tensor, mask: Tensor = None) -> Tensor:
        queries = rearrange(self.queries(x), "b n (h d) -> b h n d", h=self.num_heads)
        keys = rearrange(self.keys(x), "b n (h d) -> b h n d", h=self.num_heads)
        values = rearrange(self.values(x), "b n (h d) -> b h n d", h=self.num_heads)
        energy = torch.einsum('bhqd, bhkd -> bhqk', queries, keys)  
        if mask is not None:
            fill_value = torch.finfo(torch.float32).min
            energy.mask_fill(~mask, fill_value)

        scaling = self.emb_size ** (1 / 2)
        att = F.softmax(energy / scaling, dim=-1)
        att = self.att_drop(att)
        out = torch.einsum('bhal, bhlv -> bhav ', att, values)
        out = rearrange(out, "b h n d -> b n (h d)")
        out = self.projection(out)
        return out


class ResidualAdd(nn.Module):
    def __init__(self, fn):
        super().__init__()
        self.fn = fn

    def forward(self, x, **kwargs):
        res = x
        x = self.fn(x, **kwargs)
        x += res
        return x


class FeedForwardBlock(nn.Sequential):
    def __init__(self, emb_size, expansion, drop_p):
        super().__init__(
            nn.Linear(emb_size, expansion * emb_size),
            nn.GELU(),
            nn.Dropout(drop_p),
            nn.Linear(expansion * emb_size, emb_size),
        )


class GELU(nn.Module):
    def forward(self, input: Tensor) -> Tensor:
        return input*0.5*(1.0+torch.erf(input/math.sqrt(2.0)))


class TransformerEncoderBlock(nn.Sequential):
    def __init__(self,
                 emb_size,
                 num_heads=10,
                 drop_p=0.5,
                 forward_expansion=4,
                 forward_drop_p=0.5):
        super().__init__(
            ResidualAdd(nn.Sequential(
                nn.LayerNorm(emb_size),
                MultiHeadAttention(emb_size, num_heads, drop_p),
                nn.Dropout(drop_p)
            )),
            ResidualAdd(nn.Sequential(
                nn.LayerNorm(emb_size),
                FeedForwardBlock(
                    emb_size, expansion=forward_expansion, drop_p=forward_drop_p),
                nn.Dropout(drop_p)
            )
            ))


class TransformerEncoder(nn.Sequential):
    def __init__(self, depth, emb_size):
        super().__init__(*[TransformerEncoderBlock(emb_size) for _ in range(depth)])

class Conformer(nn.Module):
    def __init__(self , emb_size=40, depth=2,sub_n=1,ch = 59,output_dim=output_dim, **kwargs):
        super().__init__()
        self.patchem = PatchEmbedding(emb_size,sub_n,ch=ch)
        self.trans = TransformerEncoder(depth, emb_size)
        self.head = None
        self.output_dim = output_dim

    def forward(self, x,subid, **kwargs):
        x = self.patchem(x,subid)
        x = self.trans(x)
        # 第一次前向传播时初始化 MLP 头
        # feature_g = x.mean(dim=1)
        feature_g = torch.flatten(x, start_dim=1)
        if self.head is None:
            input_dim = feature_g.shape[1]
            output_dim = self.output_dim   
            
            self.head = nn.Sequential(
                nn.Linear(input_dim, 256), 
                nn.ELU(),                 
                nn.Dropout(0.5),           
                nn.Linear(256, 32),
                nn.ELU(),
                nn.Dropout(0.3),
                nn.Linear(32, output_dim)
            ).to(feature_g.device)
        output = self.head(feature_g)
        return x,output

eegencoder = Conformer(sub_n = sub_n,ch = n_eeg_ch)

In [ ]:
batch = next(train_with_feat_loader_iter)
clip_text, m_tokens, m_tokens_len,eeg,target_feat,subid = batch
out = eegencoder(eeg,subid)
eeg_f_dim = out[0].shape[-1]
clip_dim = out[1].shape[-1]
eeg.shape,out[0].shape,out[1].shape,eeg_f_dim,clip_dim

# EEG2MOTION model

In [ ]:
vqvae = get_vqvae(args, is_upper_edit=False)
ckpt = torch.load(args.resume_pth, map_location='cpu')
vqvae.load_state_dict(ckpt['net'], strict=True)
vqvae.eval()
vqvae.cuda()

In [ ]:
def load_matching_weights(model, pretrained_state_dict):
    model_dict = model.state_dict()

    matched_dict = {
        k: v
        for k, v in pretrained_state_dict.items()
        if k in model_dict and model_dict[k].shape == v.shape
    }

    print(f"Loaded {len(matched_dict)} parameters")

    missing = [
        k for k in pretrained_state_dict
        if k not in matched_dict
    ]

    if len(missing) > 0:
        print("Skipped:")
        for k in missing:
            print(k)

    model_dict.update(matched_dict)
    model.load_state_dict(model_dict)

    return model

In [ ]:
from models.e2m_trans import EEG2Motion_Transformer 
if args.motion_generation_method == 'from_scratch':
    eeg2motion = EEG2Motion_Transformer(vqvae,
                                    eegencoder,
                                    num_vq=args.nb_code, 
                                    embed_dim=args.eeg_embed_dim, 
                                    clip_dim=clip_dim,
                                    eeg_f_dim=eeg_f_dim,
                                    block_size=args.block_size, 
                                    num_layers=args.eeg_num_layers, 
                                    num_local_layer=args.eeg_num_local_layer, 
                                    n_head=args.eeg_n_head, 
                                    drop_out_rate=args.eeg_drop_out_rate, 
                                    fc_rate=args.eeg_ff_rate
                                    )
elif args.motion_generation_method == 'finetuning':
    eeg2motion = EEG2Motion_Transformer(vqvae,
                                    eegencoder,
                                    num_vq=args.nb_code, 
                                    embed_dim=args.embed_dim_gpt, 
                                    clip_dim=clip_dim,
                                    eeg_f_dim=eeg_f_dim,
                                    block_size=args.block_size, 
                                    num_layers=args.num_layers, 
                                    num_local_layer=args.num_local_layer, 
                                    n_head=args.n_head_gpt, 
                                    drop_out_rate=args.drop_out_rate, 
                                    fc_rate=args.ff_rate
                                    )
    load_matching_weights(
        eeg2motion.trans_base,
        mmm.maskdecoder.trans_base.state_dict()
    )
    load_matching_weights(
        eeg2motion.trans_head,
        mmm.maskdecoder.trans_head.state_dict()
    )

# Training

In [ ]:
def clip_loss_cosine(new_embeds, frozen_embeds, temperature=0.07):
    new_embeds = F.normalize(new_embeds, p=2, dim=-1)
    frozen_embeds = F.normalize(frozen_embeds, p=2, dim=-1)
    
    logits = torch.matmul(new_embeds, frozen_embeds.T) / temperature
    
    labels = torch.arange(len(logits)).to(logits.device)
    
    loss_new = F.cross_entropy(logits, labels)
    loss_frozen = F.cross_entropy(logits.T, labels)
    
    return (loss_new + loss_frozen) / 2

def contrastive_loss_dist(new_embeds, frozen_embeds, margin=10.0):
    """
    根据 HumanML3D 论文实现的对比损失函数 。
    
    参数:
    - new_embeds: 当前模态的特征向量 (s)
    - frozen_embeds: 对应模态的特征向量 (m)
    - margin: 负样本对的边界距离，论文中设为 10 
    """
    # 1. 计算所有对的 Euclidean 距离矩阵 (Ds,m = ||s - m||2) 
    # dist[i, j] 表示第 i 个文本和第 j 个动作之间的 L2 距离
    dist = torch.cdist(new_embeds, frozen_embeds, p=2) 
    
    batch_size = new_embeds.size(0)
    # 构造掩码：对角线为匹配对 (y=0)，非对角线为不匹配对 (y=1) 
    # 注意：文章定义 y=0 为匹配，y=1 为不匹配
    mask_matched = torch.eye(batch_size, device=new_embeds.device)
    mask_mismatched = 1 - mask_matched
    
    # 2. 计算匹配对损失 (y=0): (Ds,m)^2 
    # 只取对角线上的距离平方
    loss_matched = (dist.pow(2) * mask_matched).sum() / batch_size
    
    # 3. 计算不匹配对损失 (y=1): {max(0, m - Ds,m)}^2 
    # 强制不匹配对的距离至少为 margin (10) 
    loss_mismatched = (torch.clamp(margin - dist, min=0).pow(2) * mask_mismatched).sum() / (batch_size * (batch_size - 1))
    
    # 返回总损失
    return loss_matched + loss_mismatched

In [ ]:
def cosine_similarity_matrix(matrix1, matrix2):
    """
        Params:
        -- matrix1: N1 x D
        -- matrix2: N2 x D
        Returns:
        -- sim: N1 x N2
        sim[i, j] == cosine_similarity(matrix1[i], matrix2[j])
    """
    assert matrix1.shape[1] == matrix2.shape[1]
    matrix1_norm = matrix1 / np.linalg.norm(matrix1, axis=1, keepdims=True)
    matrix2_norm = matrix2 / np.linalg.norm(matrix2, axis=1, keepdims=True)
    sim_matrix = np.dot(matrix1_norm, matrix2_norm.T)
    return sim_matrix


def calculate_top_k_from_similarity(mat, top_k):
    """
        mat: 相似度排序后的索引矩阵，shape (N, N)
             注意：相似度是从大到小排序的
        top_k: int
        Returns:
        -- top_k_mat: shape (N, top_k)
    """
    size = mat.shape[0]
    gt_mat = np.expand_dims(np.arange(size), 1).repeat(size, 1)
    bool_mat = (mat == gt_mat)
    correct_vec = False
    top_k_list = []
    for i in range(top_k):
        correct_vec = (correct_vec | bool_mat[:, i])
        top_k_list.append(correct_vec[:, None])
    top_k_mat = np.concatenate(top_k_list, axis=1)
    return top_k_mat


def calculate_R_precision_cosine(embedding1, embedding2, top_k, sum_all=False):
    """
        基于余弦相似度计算R-precision
        Params:
        -- embedding1: N x D
        -- embedding2: N x D
        -- top_k: int
        -- sum_all: bool, 是否返回累加结果
        Returns:
        -- top_k_mat or sum_vec: 召回矩阵或其累加向量
        -- matching_score: 对角线的相似度之和
    """
    sim_mat = cosine_similarity_matrix(embedding1, embedding2)
    matching_score = np.trace(sim_mat)  # 对角线的相似度之和
    
    # 相似度越高越匹配，所以用降序排列（argsort默认升序，取负值或加[::-1]）
    argmax = np.argsort(-sim_mat, axis=1)  # 或者 np.argsort(sim_mat, axis=1)[:, ::-1]
    
    top_k_mat = calculate_top_k_from_similarity(argmax, top_k)
    
    if sum_all:
        return top_k_mat.sum(axis=0), matching_score
    else:
        return top_k_mat, matching_score


# 如果需要带温度系数的版本（类似CLIP）
def calculate_R_precision_cosine_with_temp(embedding1, embedding2, top_k, logit_scale=14.28, sum_all=False):
    """
        带温度系数的余弦相似度版本
        logit_scale: 相当于 1/temperature，CLIP中常用14.28（对应temp=0.07）
    """
    assert embedding1.shape[0] == embedding2.shape[0]
    assert embedding1.shape[1] == embedding2.shape[1]
    
    emb1_norm = embedding1 / np.linalg.norm(embedding1, axis=1, keepdims=True)
    emb2_norm = embedding2 / np.linalg.norm(embedding2, axis=1, keepdims=True)
    
    # 计算相似度矩阵并乘以logit_scale
    sim_mat = np.dot(emb1_norm, emb2_norm.T) * logit_scale
    matching_score = np.trace(sim_mat)
    
    # 相似度从大到小排序
    argmax = np.argsort(-sim_mat, axis=1)
    
    top_k_mat = calculate_top_k_from_similarity(argmax, top_k)
    
    if sum_all:
        return top_k_mat.sum(axis=0), matching_score
    else:
        return top_k_mat, matching_score

In [ ]:
if isinstance(args.eeg_data_root, list):
    # 生成组合名称，例如 "20260416S1_20260420S2_20260423S3_20260430S4"
    folder_name = "_".join([os.path.basename(path) for path in args.eeg_data_root])
else:
    folder_name = os.path.basename(args.eeg_data_root)

# 拼接保存路径
save_path_best = f'eeg_model/{folder_name}_temp.pth'
save_path_last = f'eeg_model/{folder_name}_temp.pth'
save_path_best,save_path_last

In [ ]:
logger = setup_logger(log_name=folder_name+'_temp_')
logger.info("=== Model Parameters ===")
for arg, value in vars(args).items():
    logger.info(f"  {arg}: {value}")
logger.info("========================")

In [ ]:
if args.motion_generation_method=='finetuning':
    from peft import LoraConfig, get_peft_model

    # lora_config = LoraConfig(
    #     r=8,
    #     lora_alpha=16,
    #     target_modules=[
    #         "query",
    #         "key",
    #         "value",
    #         "proj"
    #     ],
    #     lora_dropout=0.1,
    #     bias="none"
    # )
    lora_config = LoraConfig(
        r=4,
        lora_alpha=16,
        target_modules=["query", "value"],
        lora_dropout=0.05,
        bias="none"
    )

    eeg2motion = get_peft_model(eeg2motion, lora_config)

    for name, p in eeg2motion.named_parameters():
        if any(keyword in name for keyword in ["eegencoder", "cond_emb", "word_emb", "cross_att"]):
            p.requires_grad = True

    eeg2motion.print_trainable_parameters()

In [ ]:
eeg2motion = eeg2motion.cuda()
eeg2motion = torch.nn.DataParallel(eeg2motion)
eeg2motion.train()

In [ ]:
def generate_src_mask(T, length):
    B = len(length)
    mask = torch.arange(T).repeat(B, 1).to(length.device) < length.unsqueeze(-1)
    return mask
def get_model(model):
    if hasattr(model, 'module'):
        return model.module
    return model
def soft_targets(logits, temperature=3.0):
    return F.softmax(logits / temperature, dim=-1)
def kd_loss(student_logits, teacher_logits, temperature):
    probs_teacher = soft_targets(teacher_logits, temperature)
    probs_student = soft_targets(student_logits, temperature)
    loss = F.kl_div(
        probs_student.log(),  # 学生模型的 log-probabilities
        probs_teacher,        # 教师模型的概率（作为目标）
        reduction='batchmean'  # 对 batch 和 token 求平均
    )
    return loss

##### ---- Optimizer & Scheduler ---- #####
optimizer = utils_model.initial_optim(args.decay_option, args.lr, args.weight_decay, eeg2motion, args.optimizer)
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=args.eeg_milestones, gamma=args.gamma)


# for nb_iter in tqdm(range(1, args.total_iter + 1), position=0, leave=True):

bestTop3 = 0
criterion_cls = torch.nn.CrossEntropyLoss().cuda()
for nb_iter in range(1, args.eeg_iter+1):
    eeg2motion.train()
    batch = next(train_with_feat_loader_iter)
    clip_text, m_tokens, m_tokens_len,eeg,target_feat,subid = batch
    m_tokens = m_tokens.cuda()
    target_feat = target_feat.cuda()
    if args.random_level:
        indices = torch.randperm(len(clip_text))
        eeg = eeg[indices]
    eeg = eeg.float().cuda()
    m_tokens_len = torch.clamp(m_tokens_len, max=25).cuda()
    bs = m_tokens.shape[0]
    target = m_tokens    # (bs, 26)
    target = target.cuda()
    batch_size, max_len = target.shape[:2]

    text = clip.tokenize(clip_text, truncate=True).cuda()

    feat_clip_text, word_emb = clip_model(text)

    # [INFO] Swap input tokens
    if args.pkeep == -1:
        proba = np.random.rand(1)[0]
        mask = torch.bernoulli(proba * torch.ones(target.shape,
                                                device=target.device))
    else:
        mask = torch.bernoulli(args.pkeep * torch.ones(target.shape,
                                                device=target.device))
    seq_mask_no_end = generate_src_mask(max_len, m_tokens_len)
    mask = torch.logical_or(mask, ~seq_mask_no_end).int()
    r_indices = torch.randint_like(target, args.nb_code)
    input_indices = mask*target+(1-mask)*r_indices

    # Time step masking
    mask_id = get_model(mmm.vqvae).vqvae.num_code + 2
    # rand_mask_probs = torch.zeros(batch_size, device = m_tokens_len.device).float().uniform_(0.5, 1)
    rand_mask_probs = torch.zeros(batch_size, device = m_tokens_len.device).float().uniform_(args.mask_prob, 1)
    num_token_masked = (m_tokens_len * rand_mask_probs).round().clamp(min = 1)
    seq_mask = generate_src_mask(max_len, m_tokens_len+1)
    batch_randperm = torch.rand((batch_size, max_len), device = target.device) - seq_mask_no_end.int()
    batch_randperm = batch_randperm.argsort(dim = -1)
    mask_token = batch_randperm < rearrange(num_token_masked, 'b -> b 1')

    # masked_target = torch.where(mask_token, input=input_indices, other=-1)
    masked_input_indices = torch.where(mask_token, mask_id, input_indices)
    att_txt = None
    if args.teacher>0.:
        mmm.eval()
        with torch.no_grad():
            cls_pred_mmm = mmm.maskdecoder(masked_input_indices, feat_clip_text, src_mask = seq_mask, att_txt=att_txt, word_emb=word_emb)[:, 1:]

    out = eeg2motion(masked_input_indices, eeg,subid, src_mask=seq_mask, att_txt=att_txt)
    cls_pred = out[0][:, 1:]
    out_g = out[1]

    weights = seq_mask_no_end / (seq_mask_no_end.sum(-1).unsqueeze(-1) * seq_mask_no_end.shape[0])
    cls_pred_seq_masked = cls_pred[seq_mask_no_end, :].view(-1, cls_pred.shape[-1])
    target_seq_masked = target[seq_mask_no_end]
    weight_seq_masked = weights[seq_mask_no_end]
    loss_cls = F.cross_entropy(cls_pred_seq_masked, target_seq_masked, reduction = 'none')
    loss_cls = (loss_cls * weight_seq_masked).sum()   
    if args.teacher>0.:
        cls_pred_seq_masked_mmm = cls_pred_mmm[seq_mask_no_end, :].view(-1, cls_pred.shape[-1])
        loss_teacher = kd_loss(cls_pred_seq_masked,cls_pred_seq_masked_mmm,temperature=args.teacher_temp)
    else:
        loss_teacher = 0.

    target_loss = 0
    
    if args.clip_weight>0.:
        # break
        if args.clip_loss_type == 'dist':
            target_loss = contrastive_loss_dist(out_g,target_feat)
        elif args.clip_loss_type == 'cosine':
            target_loss = clip_loss_cosine(out_g,target_feat,args.clip_temp)

    pose_clip_loss = 0.
    m_token_length=25
    if args.pose_clip_weight>0.:
        eval_wrapper.movement_encoder.train()
        eval_wrapper.motion_encoder.train()
        probs = F.gumbel_softmax(
            cls_pred[:, :m_token_length],
            tau=1.,
            hard=False,
            dim=-1
        )
        pred_pose = mmm.vqvae(
            probs,
            type='decode_soft'
        )
        movements = eval_wrapper.movement_encoder(pred_pose[..., :-4])
        m_lens = torch.full_like(m_tokens_len, 25).cuda()
        pred_motion_embedding = eval_wrapper.motion_encoder(movements, m_lens)
        pose_clip_loss = contrastive_loss_dist(pred_motion_embedding,target_feat)
    
    loss = loss_cls*(1-args.teacher) + loss_teacher*args.teacher + args.clip_weight*target_loss + args.pose_clip_weight*pose_clip_loss
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()

    if nb_iter%args.val_per_epoch == 0:
        eeg2motion.eval()
        eval_wrapper.movement_encoder.eval()
        eval_wrapper.motion_encoder.eval()
        current_lr = optimizer.param_groups[0]['lr']
        target_temp_R = 0
        pose_temp_R = 0
        nb_sample = 0
        with torch.no_grad():
            for eval_iters in range(10):
                batch = next(val_with_feat_loader_iter)
                clip_text, m_tokens, m_tokens_len,eeg,target_feat,subid = batch
                target_feat = target_feat.cuda()
                eeg = eeg.float().cuda()

                blank_id = eeg2motion.module.num_vq
                lengths = torch.tensor([100]*eeg.shape[0]).cuda()
                m_token_length = 25
                index_motion,_,out_g = eeg2motion(eeg,subid, type="sample", m_length=lengths, rand_pos=True,if_test=False)
                pred_pose_eeg = torch.zeros((len(clip_text), 196, 263)).cuda()
                for k in range(len(clip_text)):
                    pred_pose = mmm.vqvae(index_motion[k:k+1, :m_token_length], type='decode')
                    pred_pose_eeg[k:k+1, :m_token_length*4] = pred_pose
                movements = eval_wrapper.movement_encoder(pred_pose_eeg[..., :-4]).detach()
                m_lens = torch.full_like(m_tokens_len, 25).cuda()
                pred_motion_embedding = eval_wrapper.motion_encoder(movements, m_lens)
                movements = eval_wrapper.movement_encoder(m_tokens[..., :-4].detach().cuda().float())
                m_lens = m_tokens_len // eval_wrapper.opt.unit_length
                true_motion_embedding = eval_wrapper.motion_encoder(movements, m_lens.cuda())
                p_temp_R, temp_match = calculate_R_precision(true_motion_embedding.detach().cpu().numpy(), pred_motion_embedding.detach().cpu().numpy(), top_k=3, sum_all=True)
                pose_temp_R += p_temp_R

                out_g = out_g.squeeze()
                if args.clip_loss_type == 'dist':
                    clip_loss_test = args.clip_weight*contrastive_loss_dist(out_g,target_feat,args.clip_temp)
                elif args.clip_loss_type == 'cosine':
                    clip_loss_test = args.clip_weight*clip_loss_cosine(out_g,target_feat,args.clip_temp)

                temp_R = np.array([0,0,0])
                if args.clip_weight>0.:
                    temp_R, temp_match = calculate_R_precision_cosine_with_temp(out_g.detach().cpu().numpy(), target_feat.detach().cpu().numpy(), top_k=3, sum_all=True)
                target_temp_R += temp_R
                
                nb_sample += len(clip_text)
            pose_temp_R = pose_temp_R/nb_sample
            target_temp_R = target_temp_R/nb_sample

            top3mean = pose_temp_R[2]
            if top3mean > bestTop3:
                logger.info('new best top3: %.6f' % top3mean)
                bestTop3 = top3mean
                torch.save(eeg2motion.module.state_dict(), save_path_best)
                
        logger.info('Epoch: %d - Clip loss: %.6f - ClipTop1: %.6f - ClipTop2: %.6f - ClipTop3: %.6f - PTop1: %.6f - PTop2: %.6f - PTop3: %.6f' % (
            nb_iter, 
            loss.detach().cpu().numpy(),
            target_temp_R[0],
            target_temp_R[1],
            target_temp_R[2],
            pose_temp_R[0],
            pose_temp_R[1],
            pose_temp_R[2]
        ))

In [ ]:
torch.save(eeg2motion.module.state_dict(), save_path_last)

In [ ]:
eeg2motion.module.load_state_dict(torch.load(save_path_best))
eeg2motion.cuda()

# Test

In [ ]:
test_batch = next(iter(test_with_feat_loader))
clip_text, m_tokens, m_tokens_len,eeg,target_feat,subid = test_batch
eeg_test = eeg.float().cuda()
blank_id = eeg2motion.module.num_vq
eeg2motion.eval()
with torch.no_grad():
    lengths = torch.tensor([100]*eeg_test.shape[0]).cuda()
    index_motion,_,eeg_fea_g = eeg2motion(eeg_test,subid, type="sample", m_length=lengths, rand_pos=True,if_test=False)
    pred_length = (index_motion >= blank_id).int()
    pred_length = torch.topk(pred_length, k=1, dim=1).indices.squeeze().float()
    pred_pose_text = mmm(clip_text, lengths, rand_pos=True)

In [ ]:
lengths = 100
pred_pose_eeg = torch.zeros((len(clip_text), 196, 263)).cuda()
m_token_length = 25
for k in range(len(clip_text)):
    pred_pose = mmm.vqvae(index_motion[k:k+1, :m_token_length], type='decode')
    pred_pose_eeg[k:k+1, :lengths] = pred_pose

In [ ]:
m_tokens.shape,m_tokens_len

In [ ]:
true_pose = m_tokens

In [ ]:
id =0
std = np.load('./exit/t2m-std.npy')
mean = np.load('./exit/t2m-mean.npy')
file_name = '_temp'
fig = visualize_2motions(pred_pose_eeg[id].detach().cpu().numpy(), std, mean, 't2m', lengths, save_path='./output/'+file_name+'.html')

In [ ]:
fig = visualize_2motions(pred_pose_text[0][id].detach().cpu().numpy(), std, mean, 't2m', lengths, save_path='./output/'+file_name+'.html')
clip_text[id]

In [ ]:
fig = visualize_2motions(true_pose[id].detach().cpu().numpy(), std, mean, 't2m', lengths, save_path='./output/'+file_name+'.html')
clip_text[id]

In [ ]:
import numpy as np
import torch
from scipy import stats

TOP_K_COUNT = 3
NUM_RUNS = 20
metric_keys = ['T_R', 'pose_R', 'T_match', 'pose_match', 'FID', 'Diversity']

results_store = {k: {m: [] for m in metric_keys} for k in ['real', 'shuf', 'noise']}
lengths = torch.tensor([100]*eeg_test.shape[0]).cuda()
eeg2motion.eval()
if hasattr(mmm, 'vqvae'): mmm.vqvae.eval()

diversity_gt = []
for run_idx in range(NUM_RUNS):
    run_data = {k: {m: (np.zeros(TOP_K_COUNT) if 'R' in m else 0.0) for m in metric_keys} for k in ['real', 'shuf', 'noise']}
    
    motion_annos = []
    motion_preds = {'real': [], 'noise': []}
    
    current_nb_sample = 0

    for test_batch in iter(test_with_feat_loader):
        clip_text, m_tokens, m_tokens_len, eeg, target_feat, subid = test_batch
        bs = len(clip_text)
        current_nb_sample += bs
        
        with torch.no_grad():
            eeg = eeg.float().cuda()
            target_feat_np = target_feat.detach().cpu().numpy()
            
            m_tokens_cuda = m_tokens.detach().cuda().float()
            true_movements = eval_wrapper.movement_encoder(m_tokens_cuda[..., :-4])
            true_m_lens = m_tokens_len // eval_wrapper.opt.unit_length
            true_motion_emb = eval_wrapper.motion_encoder(true_movements, true_m_lens.cuda())
            true_motion_emb_np = true_motion_emb.detach().cpu().numpy()
            motion_annos.append(true_motion_emb)

            # --- 1. Real & Shuffle Group ---
            index_motion, _, eeg_fea_g = eeg2motion(eeg, subid, type="sample", m_length=lengths, rand_pos=True, if_test=False)
            eeg_fea_np = eeg_fea_g.detach().cpu().numpy()
            
            pred_pose_eeg = torch.zeros((bs, 196, 263)).cuda()
            for k in range(bs):
                pred_pose = mmm.vqvae(index_motion[k:k+1, :m_token_length], type='decode')
                pred_pose_eeg[k:k+1, :m_token_length*4] = pred_pose
            
            movements_real = eval_wrapper.movement_encoder(pred_pose_eeg[..., :-4])
            m_lens_fixed = torch.full_like(m_tokens_len, 25).cuda()
            pred_motion_emb_real = eval_wrapper.motion_encoder(movements_real, m_lens_fixed)
            pred_motion_emb_real_np = pred_motion_emb_real.detach().cpu().numpy()
            motion_preds['real'].append(pred_motion_emb_real)

            # --- 2. Noise Group (Input Gaussian) ---
            noise_eeg = torch.randn_like(eeg).cuda()
            index_motion_n, _, eeg_fea_n = eeg2motion(noise_eeg, subid, type="sample", m_length=lengths, rand_pos=True, if_test=False)
            eeg_fea_n_np = eeg_fea_n.detach().cpu().numpy()
            
            noise_pose_eeg = torch.zeros((bs, 196, 263)).cuda()
            for k in range(bs):
                n_pose = mmm.vqvae(index_motion_n[k:k+1, :m_token_length], type='decode')
                noise_pose_eeg[k:k+1, :m_token_length*4] = n_pose
            
            movements_noise = eval_wrapper.movement_encoder(noise_pose_eeg[..., :-4])
            pred_motion_emb_noise = eval_wrapper.motion_encoder(movements_noise, m_lens_fixed)
            pred_motion_emb_noise_np = pred_motion_emb_noise.detach().cpu().numpy()
            motion_preds['noise'].append(pred_motion_emb_noise)

            # Helper for R-precision
            def get_r_and_match(p, t, type='cosine'):
                if type == 'cosine':
                    return calculate_R_precision_cosine_with_temp(p, t, top_k=TOP_K_COUNT, sum_all=True)
                return calculate_R_precision(p, t, top_k=TOP_K_COUNT, sum_all=True)

            # Real Metrics
            r, m = get_r_and_match(eeg_fea_np, target_feat_np, 'cosine')
            run_data['real']['T_R'] += r; run_data['real']['T_match'] += m
            r, m = get_r_and_match(true_motion_emb_np, pred_motion_emb_real_np, 'dist')
            run_data['real']['pose_R'] += r; run_data['real']['pose_match'] += m

            # Shuffle Metrics
            shuf_idx = np.random.permutation(bs)
            r, m = get_r_and_match(eeg_fea_np, target_feat_np[shuf_idx], 'cosine')
            run_data['shuf']['T_R'] += r; run_data['shuf']['T_match'] += m
            r, m = get_r_and_match(true_motion_emb_np, pred_motion_emb_real_np[shuf_idx], 'dist')
            run_data['shuf']['pose_R'] += r; run_data['shuf']['pose_match'] += m

            # Noise Metrics
            r, m = get_r_and_match(eeg_fea_n_np, target_feat_np, 'cosine')
            run_data['noise']['T_R'] += r; run_data['noise']['T_match'] += m
            r, m = get_r_and_match(true_motion_emb_np, pred_motion_emb_noise_np, 'dist')
            run_data['noise']['pose_R'] += r; run_data['noise']['pose_match'] += m

    # ---  FID / Diversity ---
    anno_np = torch.cat(motion_annos, dim=0).cpu().numpy()
    gt_mu, gt_cov = calculate_activation_statistics(anno_np)
    div_sample_size = 300 if current_nb_sample > 300 else 100
    diversity_gt.append(calculate_diversity(anno_np, div_sample_size))

    for group in ['real', 'noise']:
        pred_np = torch.cat(motion_preds[group], dim=0).cpu().numpy()
        mu, cov = calculate_activation_statistics(pred_np)
        run_data[group]['FID'] = calculate_frechet_distance(gt_mu, gt_cov, mu, cov)
        run_data[group]['Diversity'] = calculate_diversity(pred_np, div_sample_size)
    
    for g in ['real', 'shuf', 'noise']:
        for m in metric_keys:
            if g == 'shuf' and m in ['FID', 'Diversity']: continue
            results_store[g][m].append(run_data[g][m] / current_nb_sample if 'FID' not in m and 'Diversity' not in m else run_data[g][m])

logger.info("="*50)
logger.info(f"STATISTICAL EVALUATION OVER {NUM_RUNS} RUNS")
logger.info("="*50)

def print_stat(name, real_list, comp_list, is_array=False):
    real_arr, comp_arr = np.array(real_list), np.array(comp_list)
    r_mean, r_std = np.mean(real_arr, axis=0), np.std(real_arr, axis=0)
    c_mean, c_std = np.mean(comp_arr, axis=0), np.std(comp_arr, axis=0)
    _, p_vals = stats.ttest_rel(real_arr, comp_arr, axis=0)
    
    if not is_array:
        if p_vals < 0.001: sig = "***"
        elif p_vals < 0.01: sig = "**"
        elif p_vals < 0.05: sig = "*"
        else :sig = ""
        logger.info(f"{name:<15}: Real {r_mean:.4f}±{r_std:.4f} vs {c_mean:.4f}±{c_std:.4f} | p={p_vals:.2e} {sig}")
    else:
        for i in range(TOP_K_COUNT):
            if p_vals[i] < 0.001: sig = "***"
            elif p_vals[i] < 0.01: sig = "**"
            elif p_vals[i] < 0.05: sig = "*"
            else :sig = ""
            logger.info(f"{name}[Top-{i+1}]: Real {r_mean[i]:.4f}±{r_std[i]:.4f} vs {c_mean[i]:.4f}±{c_std[i]:.4f} | p={p_vals[i]:.2e} {sig}")

for baseline in ['shuf', 'noise']:
    logger.info(f"\n>>> Comparing Real vs {baseline.upper()}")
    print_stat("T_R_Precision", results_store['real']['T_R'], results_store[baseline]['T_R'], True)
    print_stat("Pose_R_Prec",   results_store['real']['pose_R'], results_store[baseline]['pose_R'], True)
    print_stat("Pose_Match_Score", results_store['real']['pose_match'], results_store[baseline]['pose_match'])
    if baseline == 'noise':
        print_stat("FID", results_store['real']['FID'], results_store['noise']['FID'])
        print_stat("Diversity", results_store['real']['Diversity'], results_store['noise']['Diversity'])

diversity_gt_arr = np.array(diversity_gt)
diversity_gt_arr_mean, diversity_gt_arr_std = np.mean(diversity_gt_arr, axis=0), np.std(diversity_gt_arr, axis=0)
logger.info(f"Ground Truth Diversity: {diversity_gt_arr_mean:.4f}±{diversity_gt_arr_std:.4f}")

In [ ]:
import gc
torch.cuda.empty_cache()
gc.collect()